In [1]:
import os
os.makedirs("../data/text_files",exist_ok=True)



In [86]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader , PyMuPDFLoader

dir_loader= DirectoryLoader("../data/pdf",
                            glob="**/*.pdf",
                            loader_cls=PyMuPDFLoader,
                            
                            show_progress=False)

pdf_documents= dir_loader.load()
print(pdf_documents)

[Document(metadata={'producer': 'Microsoft® Word 2024; modified using OpenPDF UNKNOWN', 'creator': 'Microsoft® Word 2024', 'creationdate': '2025-10-17T01:50:47+05:30', 'source': '..\\data\\pdf\\door 404.pdf', 'file_path': '..\\data\\pdf\\door 404.pdf', 'total_pages': 34, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-10-17T17:08:37+05:30', 'trapped': '', 'modDate': "D:20251017170837+05'30'", 'creationDate': "D:20251017015047+05'30'", 'page': 0}, page_content='DOOR 404 \n \n_warning: stay off the fourth flour'), Document(metadata={'producer': 'Microsoft® Word 2024; modified using OpenPDF UNKNOWN', 'creator': 'Microsoft® Word 2024', 'creationdate': '2025-10-17T01:50:47+05:30', 'source': '..\\data\\pdf\\door 404.pdf', 'file_path': '..\\data\\pdf\\door 404.pdf', 'total_pages': 34, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-10-17T17:08:37+05:30', 'trapped': '', 'modDate': "D:20251017170837+

In [37]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
   
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,  
        separators=["\n\n", "\n", " ", ""]  
    )
    
    
    split_docs = text_splitter.split_documents(documents)
    
    for i, doc in enumerate(split_docs):
        doc.metadata['chunk_index'] = i
        
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    return split_docs
chunks=split_documents(pdf_documents)
chunks


Split 34 documents into 48 chunks


[Document(metadata={'producer': 'Microsoft® Word 2024; modified using OpenPDF UNKNOWN', 'creator': 'Microsoft® Word 2024', 'creationdate': '2025-10-17T01:50:47+05:30', 'source': '..\\data\\pdf\\door 404.pdf', 'file_path': '..\\data\\pdf\\door 404.pdf', 'total_pages': 34, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-10-17T17:08:37+05:30', 'trapped': '', 'modDate': "D:20251017170837+05'30'", 'creationDate': "D:20251017015047+05'30'", 'page': 0, 'chunk_index': 0}, page_content='DOOR 404 \n \n_warning: stay off the fourth flour'),
 Document(metadata={'producer': 'Microsoft® Word 2024; modified using OpenPDF UNKNOWN', 'creator': 'Microsoft® Word 2024', 'creationdate': '2025-10-17T01:50:47+05:30', 'source': '..\\data\\pdf\\door 404.pdf', 'file_path': '..\\data\\pdf\\door 404.pdf', 'total_pages': 34, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-10-17T17:08:37+05:30', 'trapped': '', 'modDate':

In [14]:
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import uuid
from typing import List,Any , Dict, Tuple
from sklearn.metrics.pairwise import cosine_similarity 


c:\Users\HP\Desktop\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

      self.model_name=model_name
      self.model=None
      self._load_model()
    def _load_model(self):
        try:
           print(f"Loading model: {self.model_name}")
           self.model= SentenceTransformer(self.model_name)
           
           print(f"Model embeddings dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise 
    def embed_texts(self,texts: List[str])-> np.ndarray:
       if not self.model:
          raise ValueError("model not loaded")
       print(f"Embedding {len(texts)} texts")
       embeddings= self.model.encode(texts,show_progress_bar=True)
       print(f"Embeddings shape: {embeddings.shape}")
       return embeddings
embedding_manager= EmbeddingManager()
embedding_manager

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2702.11it/s]


Model embeddings dimension: 384


C:\Users\HP\AppData\Local\Temp\ipykernel_4476\889807041.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model embeddings dimension: {self.model.get_sentence_embedding_dimension()}")


In [48]:
class VectorStore:
    def __init__(self,collection_name: str="pdf_documents", persist_directory: str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"pdf document embeddings for RAG"})
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"existing doc in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self,documents:List[Any], embeddings:np.ndarray):
        if(len(documents)!=len(embeddings)):
            raise ValueError("Number of documents and embeddings must match")


        print(f"Adding {len(documents)} documents to vector store")

        ids=[]
        metadatas=[]
        documents_texts=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata) 
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_texts.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
 vector_store= VectorStore()
vector_store




Vector store initialized with collection: pdf_documents
existing doc in collection: 0


In [38]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2024; modified using OpenPDF UNKNOWN', 'creator': 'Microsoft® Word 2024', 'creationdate': '2025-10-17T01:50:47+05:30', 'source': '..\\data\\pdf\\door 404.pdf', 'file_path': '..\\data\\pdf\\door 404.pdf', 'total_pages': 34, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-10-17T17:08:37+05:30', 'trapped': '', 'modDate': "D:20251017170837+05'30'", 'creationDate': "D:20251017015047+05'30'", 'page': 0, 'chunk_index': 0}, page_content='DOOR 404 \n \n_warning: stay off the fourth flour'),
 Document(metadata={'producer': 'Microsoft® Word 2024; modified using OpenPDF UNKNOWN', 'creator': 'Microsoft® Word 2024', 'creationdate': '2025-10-17T01:50:47+05:30', 'source': '..\\data\\pdf\\door 404.pdf', 'file_path': '..\\data\\pdf\\door 404.pdf', 'total_pages': 34, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-10-17T17:08:37+05:30', 'trapped': '', 'modDate':

In [39]:
texts=[doc.page_content for doc in chunks]
texts

['DOOR 404 \n \n_warning: stay off the fourth flour',
 'DOOR 404 \n \n_warning: stay off the fourth flour \n \n \nTable of Contents \n[Chapter 1] — Arrival at Aarunya ................................................. Error! Bookmark not defined. \n[Chapter 2]— The missing floor .................................................... Error! Bookmark not defined. \n[Chapter 3] — The warning ............................................................................................................ 4 \n[Chapter 4] — Block C: The psychiatrist division ......................................................................... 4 \n[Chapter 5] — The fourth floor ...................................................................................................... 4 \n[Chapter 6] — A hidden button ..................................................... Error! Bookmark not defined. \n[Chapter 7] — The corridor that breathes .............................................................................

In [49]:
texts=[doc.page_content for doc in chunks]

embeddings= embedding_manager.embed_texts(texts)

vector_store.add_documents(chunks, embeddings)




Embedding 48 texts


Batches: 100%|██████████| 2/2 [00:04<00:00,  2.26s/it]


Embeddings shape: (48, 384)
Adding 48 documents to vector store
Successfully added 48 documents to vector store
Total documents in collection: 48


In [65]:
class RagRetriever:
    def __init__(self,vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.7)->List[Dict[str,Any]]:
        print(f"Retrieving documents for query: {query}")
        print(f"Top K {top_k} documents with score threshold: {score_threshold}")

        query_embedding= self.embedding_manager.embed_texts([query])[0]
        try:
            results= self.vector_store.collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k,
                
            )
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents= results['documents'][0]
                metadatas= results['metadatas'][0]
                distances= results['distances'][0]
                ids= results['ids'][0]

                for i,(doc_id,document,metadat,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    score= 1-distance
                    if score<=score_threshold:
                        retrieved_docs.append({
                            "id":doc_id,
                            "content":document,
                            "metadata":metadat,
                            "score":score,
                            "distance":distance,
                            'rank':i+1

                        })
                print(f"Retrieved {len(retrieved_docs)} documents after applying score threshold")
            else:
                print("No documents retrieved from vector store")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return[]
rag_retriever= RagRetriever(vector_store,embedding_manager)
rag_retriever

In [68]:
rag_retriever.retrieve("what is chapter 1?",top_k=5,score_threshold=0.7)

Retrieving documents for query: what is chapter 1?
Top K 5 documents with score threshold: 0.7
Embedding 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.77it/s]

Embeddings shape: (1, 384)
Retrieved 5 documents after applying score threshold


[{'id': 'doc_8e2f9d84_1',
  'content': 'DOOR 404 \n \n_warning: stay off the fourth flour \n \n \nTable of Contents \n[Chapter 1] — Arrival at Aarunya ................................................. Error! Bookmark not defined. \n[Chapter 2]— The missing floor .................................................... Error! Bookmark not defined. \n[Chapter 3] — The warning ............................................................................................................ 4 \n[Chapter 4] — Block C: The psychiatrist division ......................................................................... 4 \n[Chapter 5] — The fourth floor ...................................................................................................... 4 \n[Chapter 6] — A hidden button ..................................................... Error! Bookmark not defined. \n[Chapter 7] — The corridor that breathes ....................................................................................... 4',
 

In [87]:
def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant documents found for the query."

    prompt = f"""
Answer the following question based only on the context provided.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

In [88]:
answer = rag_simple(
    "what is chapter 1?",
    rag_retriever,
    llm,
    top_k=5
)

print(answer)

Retrieving documents for query: what is chapter 1?
Top K 5 documents with score threshold: 0.7
Embedding 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.43it/s]

Embeddings shape: (1, 384)
Retrieved 5 documents after applying score threshold


BadRequestError: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}